# Topic: File Integrity Checker - Cryptographic baseline databases, file system change notifications, and tampering alerts
 
## 1. THE DEFENSIVE ROLE OF INTEGRITY CHECKERS
- **File Integrity Monitoring (FIM)**: A Blue Team defense mechanism used to detect unauthorized modifications to system files, database configurations, and application directories.
- **Detection Logic**:
  - **Baseline Generation**: Run a sweep over target directories and record the path and cryptographic hash (SHA-256) of each file into a secure database file.
  - **Active Scan Auditing**: Periodically scan the folders again, calculating active file hashes and comparing them to the baseline database.
- **Alert Classifications**:
  1. **Modified**: The file exists at the baseline path, but its hash has changed (indicating tampering or editing).
  2. **New / Created**: The file is active on disk but not registered in the baseline database (indicating unauthorized file creations or Trojan horse insertions).
  3. **Deleted**: The file is in the baseline database but missing from disk (indicating file system damage or log deletions).
 
---


In [ ]:
import hashlib
import os
import json

class FileIntegrityChecker:
    def __init__(self, target_dir, db_file="integrity_baseline.json"):
        self.target_dir = target_dir
        self.db_file = db_file
        self.baseline = {}

    def get_file_hash(self, filepath):
        """Computes incremental SHA-256 for a file."""
        hasher = hashlib.sha256()
        try:
            with open(filepath, "rb") as f:
                while True:
                    chunk = f.read(4096)
                    if not chunk:
                        break
                    hasher.update(chunk)
            return hasher.hexdigest()
        except OSError:
            # Handle permission errors or file access locks safely
            return None

    def generate_baseline(self):
        """Sweeps target_dir and creates a new baseline database."""
        baseline = {}
        for root, _, files in os.walk(self.target_dir):
            for file in files:
                filepath = os.path.join(root, file)
                file_hash = self.get_file_hash(filepath)
                if file_hash:
                    # Store relative path to keep DB independent of root location shifts
                    rel_path = os.path.relpath(filepath, self.target_dir)
                    baseline[rel_path] = file_hash

        self.baseline = baseline
        # Save baseline database as JSON
        with open(self.db_file, "w", encoding="utf-8") as f:
            json.dump(baseline, f, indent=4)
        print(f"Baseline successfully generated for {len(baseline)} files.")

    def load_baseline(self):
        """Loads baseline database from disk."""
        if os.path.exists(self.db_file):
            with open(self.db_file, "r", encoding="utf-8") as f:
                self.baseline = json.load(f)
            return True
        return False

    def verify_integrity(self):
        """Scans folder and reports Modified, Created, or Deleted states."""
        if not self.baseline and not self.load_baseline():
            print("Error: No baseline database loaded. Please generate baseline first.")
            return

        current_files = {}
        # 1. Scan active files on disk
        for root, _, files in os.walk(self.target_dir):
            for file in files:
                filepath = os.path.join(root, file)
                file_hash = self.get_file_hash(filepath)
                if file_hash:
                    rel_path = os.path.relpath(filepath, self.target_dir)
                    current_files[rel_path] = file_hash

        # 2. Check for modifications and deletions
        modified_files = []
        deleted_files = []
        for rel_path, baseline_hash in self.baseline.items():
            if rel_path not in current_files:
                deleted_files.append(rel_path)
            elif current_files[rel_path] != baseline_hash:
                modified_files.append(rel_path)

        # 3. Check for new files (Created)
        created_files = [rel_path for rel_path in current_files if rel_path not in self.baseline]

        # Print alerts report
        print("\n--- File Integrity Audit Report ---")
        if not modified_files and not created_files and not deleted_files:
            print("Audit Status: SECURE. No file system changes detected.")
            return True

        for item in deleted_files:
            print(f"[!] ALERT: File DELETED  -> {item}")
        for item in modified_files:
            print(f"[!] ALERT: File MODIFIED -> {item} (Hash mismatch!)")
        for item in created_files:
            print(f"[!] ALERT: New File CREATED -> {item}")
        return False



In [ ]:
# Demonstrating the Integrity Checker in action
print("--- Creating Lab Directory ---")
lab_dir = "fim_sandbox"
os.makedirs(lab_dir, exist_ok=True)

# Create two initial configuration files
with open(os.path.join(lab_dir, "config.ini"), "w") as f:
    f.write("setting=true\n")
with open(os.path.join(lab_dir, "system.log"), "w") as f:
    f.write("Log initialized.\n")

# Instantiate and generate baseline
checker = FileIntegrityChecker(lab_dir)
checker.generate_baseline()

# Verify (should be secure)
checker.verify_integrity()



In [ ]:
# Tamper simulation
print("\n--- Simulating Tampering ---")
# 1. Modify config.ini
with open(os.path.join(lab_dir, "config.ini"), "w") as f:
    f.write("setting=false\n")  # Altered setting

# 2. Create unauthorized file
with open(os.path.join(lab_dir, "trojan.py"), "w") as f:
    f.write("exec(payload)\n")

# Run verify integrity sweep again
checker.verify_integrity()



In [ ]:
# Clean up sandbox
for root, _, files in os.walk(lab_dir):
    for file in files:
        os.remove(os.path.join(root, file))
os.rmdir(lab_dir)
if os.path.exists("integrity_baseline.json"):
    os.remove("integrity_baseline.json")
